# Loan Approval Prediction — Data Audit & EDA
This notebook performs an initial audit of the raw dataset followed by exploratory data analysis (EDA) to understand the data structure, identify missing values, classify features, and assess which features are likely to be predictive of loan approval.

In [1]:
import pandas as pd

## 1. Importing Libraries

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [3]:
# 1. Data Loading
df= pd.read_csv("../data/loan_data_set.csv")

# 2. Data Audit

## 2.1 First 5 data

In [4]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


### 2.2 Dataset shape

In [5]:
df.shape

(614, 13)

### Findings
- Dataset contains 614 rows and 13 columns.
- Dataset is relatively small.
- Cross-validation will be important during modeling.

### 2.3 Data Types

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


### Findings
- What are the data types
- How many data are present for a certain column

### 2.4 Missing Values

In [7]:
df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

### Findings

- Missing values exist in Gender, Married, Dependents,
  Self_Employed, LoanAmount, Loan_Amount_Term and Credit_History.
- Credit_History contains the highest number of missing values.
- Missing-value handling must be performed before training.

### 2.5 Feature Classification

In [8]:
df.nunique()

Loan_ID              614
Gender                 2
Married                2
Dependents             4
Education              2
Self_Employed          2
ApplicantIncome      505
CoapplicantIncome    287
LoanAmount           203
Loan_Amount_Term      10
Credit_History         2
Property_Area          3
Loan_Status            2
dtype: int64

### Findings

Identifier:
- Loan_ID

Binary Categorical:
- Gender
- Married
- Education
- Self_Employed
- Credit_History

Ordinal Categorical:
- Dependents

Nominal Categorical:
- Property_Area

Numerical:
- ApplicantIncome
- CoapplicantIncome
- LoanAmount
- Loan_Amount_Term

### 2.6 Target Distibution

In [9]:
df["Loan_Status"].value_counts()

Loan_Status
Y    422
N    192
Name: count, dtype: int64

### Findings
- Approved loans (Y): 422
- Rejected loans (N): 192
- Target distribution is moderately imbalanced.
- Accuracy alone should not be used for model evaluation.

# 3. EDA

### 3.1 Credit History vs Loan Status
Credit history is flagged as a strong predictor based on the EDA summary. This crosstab shows the raw count of approvals and rejections by credit history value.

In [10]:
pd.crosstab(df["Credit_History"], df["Loan_Status"])

Loan_Status,N,Y
Credit_History,,
0.0,82,7
1.0,97,378


### Findings

- Applicants with Credit_History = 0 have an approval rate of approximately 8%.
- Applicants with Credit_History = 1 have an approval rate of approximately 80%.
- This large difference suggests that Credit_History has a strong relationship with loan approval.
- Credit_History is likely to be an important predictive feature during model training.

### 3.2 Applicant Income vs Loan Status
Grouped statistics to check if income distribution differs meaningfully between approved and rejected applicants.

In [11]:
df.groupby("Loan_Status")["ApplicantIncome"].describe()

,count,mean,std,min,25%,50%,75%,max
Loan_Status,,,,,,,,
N,192.0,5446.078125,6819.558528,150.0,2885.0,3833.5,5861.25,81000.0
Y,422.0,5384.068720,5765.441615,210.0,2877.5,3812.5,5771.50,63337.0


### Findings
- ApplicantIncome alone cannot help achieve the target as the mean and median are almost same
- We do not consider std here because of the outliers the value comes to be very large.
- Max and min also doesnot determine the outcome as they are just outliers.

### 3.3 Loan Amount vs Loan Status
Grouped statistics to check if the requested loan amount differs between approved and rejected applicants.

In [12]:
df.groupby("Loan_Status")["LoanAmount"].describe()

,count,mean,std,min,25%,50%,75%,max
Loan_Status,,,,,,,,
N,181.0,151.220994,85.862783,9.0,100.0,129.0,176.0,570.0
Y,411.0,144.294404,85.484607,17.0,100.0,126.0,161.0,700.0


### Findings
- Median difference is ~3 unit
- Mean difference is ~7 unit
- LoanAmount alone does not appear to strongly distinguish approved and rejected applicants.

### 3.4 Property Area vs Loan Status
Crosstab to assess whether location (Urban / Semiurban / Rural) is associated with approval rates.

In [14]:
pd.crosstab(df["Property_Area"],df["Loan_Status"])

Loan_Status,N,Y
Property_Area,,
Rural,69,110
Semiurban,54,179
Urban,69,133


### Findings

- Semiurban applicants have the highest approval rate (~76.8%).
- Urban applicants have an approval rate of ~65.8%.
- Rural applicants have the lowest approval rate (~61.5%).
- Property_Area appears to have some relationship with loan approval.
- The relationship is weaker than Credit_History but stronger than what we observed for ApplicantIncome and LoanAmount.

### 3.5 Education vs Loan Status
Crosstab to assess whether education level (Graduate / Not Graduate) is associated with approval rates.

In [15]:
pd.crosstab(df["Education"],df["Loan_Status"])

Loan_Status,N,Y
Education,,
Graduate,140,340
Not Graduate,52,82


### Findings
- Education appears to be associated with higher approval rates.
- Graduate ≈ 70% approval
- Non Graduate ≈ 61.1% approval
- Difference ~9 units, difference is not very huge but is meaningful

## Feature Signal Ranking (Based on Initial EDA)
### Strong Signal
- Credit_History
### Moderate Signal
- Property_Area
- Education
### Weak Signal
- LoanAmount
- ApplicantIncome

In [19]:
df["Credit_History"].value_counts(dropna=False)

Credit_History
1.0    475
0.0     89
NaN     50
Name: count, dtype: int64

### 3.6 Gender vs Loan Status
Crosstab to check if gender shows a meaningful difference in approval rates.

In [23]:
pd.crosstab(df["Gender"],df["Loan_Status"])

Loan_Status,N,Y
Gender,,
Female,37,75
Male,150,339


In [25]:
df.isnull().sum()

Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

### 3.7 Loan Amount Term vs Loan Status
Grouped statistics to check if loan term length differs between approved and rejected applicants.

In [31]:
df.groupby("Loan_Status")["Loan_Amount_Term"].describe()

,count,mean,std,min,25%,50%,75%,max
Loan_Status,,,,,,,,
N,186.0,344.064516,69.238921,36.0,360.0,360.0,360.0,480.0
Y,414.0,341.072464,63.247770,12.0,360.0,360.0,360.0,480.0


In [33]:
df["Loan_Amount_Term"].describe()

count    600.00000
mean     342.00000
std       65.12041
min       12.00000
25%      360.00000
50%      360.00000
75%      360.00000
max      480.00000
Name: Loan_Amount_Term, dtype: float64

### 3.8 Self Employment vs Loan Status
Crosstab to assess whether self-employment status is associated with approval rates.

In [36]:
pd.crosstab(df["Self_Employed"],df["Loan_Status"])

Loan_Status,N,Y
Self_Employed,,
No,157,343
Yes,26,56
